# 2. SQL Analysis

**Project:** HR Analytics: Employee Attrition  

---

## Overview
Business-framed SQL queries using DuckDB against the raw CSV. All queries are cross-validated against pandas equivalents.

## Contents
1. Setup & DuckDB Registration
2. Basic Information
3. Turnover Rate by Segment
4. Key Findings

---
## 1. Setup & DuckDB Registration

In [2]:
import duckdb
import pandas as pd

# Connect to db and register the CSV as a virtual table
con = duckdb.connect()
con.execute("CREATE VIEW hr AS SELECT * FROM '../data/raw/aug_train.csv'")

# Data load via pandas for cross-checks
train = pd.read_csv('../data/raw/aug_train.csv')
print('DuckDB view registered. Row count:')
con.execute('SELECT COUNT(*) AS rows FROM hr').fetchdf()

DuckDB view registered. Row count:


,rows
0,19158


---
## 2. Basic Information

### Total row count

First, let's check how many rows of data we have.

In [3]:
con.execute('SELECT COUNT(*) AS total_rows FROM hr').fetchdf()

,total_rows
0,19158


### Target class distribution
Let's check how balanced the distribution of target values are.

In [4]:
target_distr = """
SELECT
    target,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM hr
GROUP BY target
ORDER BY target
"""
con.execute(target_distr).fetchdf()

,target,count,pct
0,0.0,14381,75.07
1,1.0,4777,24.93


About 24.7% of candidates are actively seeking a job change.

### Missing values
Let's find out which columns need imputation or special handling.

In [5]:
missing_values = """
SELECT
    SUM(CASE WHEN gender           IS NULL THEN 1 ELSE 0 END) AS gender_missing,
    SUM(CASE WHEN company_type     IS NULL THEN 1 ELSE 0 END) AS company_type_missing,
    SUM(CASE WHEN company_size     IS NULL THEN 1 ELSE 0 END) AS company_size_missing,
    SUM(CASE WHEN major_discipline IS NULL THEN 1 ELSE 0 END) AS major_discipline_missing,
    SUM(CASE WHEN education_level  IS NULL THEN 1 ELSE 0 END) AS education_level_missing,
    SUM(CASE WHEN experience       IS NULL THEN 1 ELSE 0 END) AS experience_missing,
    SUM(CASE WHEN last_new_job     IS NULL THEN 1 ELSE 0 END) AS last_new_job_missing,
    COUNT(*) AS total_rows
FROM hr
"""
con.execute(missing_values).fetchdf()

,gender_missing,company_type_missing,company_size_missing,major_discipline_missing,education_level_missing,experience_missing,last_new_job_missing,total_rows
0,4508.0,6140.0,5938.0,2813.0,460.0,65.0,423.0,19158


We can see that there are 6140 missing values(32%) in company_type column and 5938 (31%) in company_size column.

---
## 3. Turnover Rate by Segment

### Turnover intent by education level
Let's check if education contributes to job-switching.
**Hypothesis:** Graduate candidates are most mobile due to high numbers and early career stage.

In [6]:
turnover_education = """
SELECT
    COALESCE(education_level, 'Unknown') AS education_level,
    COUNT(*) AS total_candidates,
    SUM(target) AS seeking_change,
    ROUND(AVG(target) * 100, 2) AS turnover_rate_pct
FROM hr
GROUP BY education_level
ORDER BY turnover_rate_pct DESC
"""
result_turnover_education = con.execute(turnover_education).fetchdf()
display(result_turnover_education)

# Cross-check against pandas
check = (train.assign(education_level=train['education_level'].fillna('Unknown'))
              .groupby('education_level')['target']
              .agg(total='count', seekers='sum', rate='mean')
              .assign(rate=lambda d: (d['rate']*100).round(2))
              .sort_values('rate', ascending=False))
print('\nPandas cross-check:')
display(check)

,education_level,total_candidates,seeking_change,turnover_rate_pct
0,Graduate,11598,3245.0,27.98
1,Unknown,460,104.0,22.61
2,Masters,4361,935.0,21.44
3,High School,2017,394.0,19.53
4,Phd,414,58.0,14.01
5,Primary School,308,41.0,13.31



Pandas cross-check:


,total,seekers,rate
education_level,,,
Graduate,11598,3245.0,27.98
Unknown,460,104.0,22.61
Masters,4361,935.0,21.44
High School,2017,394.0,19.53
Phd,414,58.0,14.01
Primary School,308,41.0,13.31


Candidates with Graduate level education show the highest seeking rate. PhD holders are among the most stable (lowest rate), likely due to deep specialization making lateral moves harder.

### Turnover rate by company type
Let's examine the correlation between certain company types and turnover.

In [7]:
comp_type_turnover = """
SELECT
    COALESCE(company_type, 'Unknown') AS company_type,
    COUNT(*) AS total_candidates,
    SUM(target) AS seeking_change,
    ROUND(AVG(target) * 100, 2) AS turnover_rate_pct
FROM hr
GROUP BY company_type
ORDER BY turnover_rate_pct DESC
"""
con.execute(comp_type_turnover).fetchdf()

,company_type,total_candidates,seeking_change,turnover_rate_pct
0,Unknown,6140,2384.0,38.83
1,Other,121,29.0,23.97
2,Early Stage Startup,603,142.0,23.55
3,Public Sector,955,210.0,21.99
4,NGO,521,97.0,18.62
5,Pvt Ltd,9817,1775.0,18.08
6,Funded Startup,1001,140.0,13.99


The 'Unknown' group (missing company type — likely freelancers/consultants) has the highest seeking rate. Among known types, early-stage startups show elevated rates.

### Turnover rate by company size

In [8]:
company_size_turnover = """
SELECT
    COALESCE(company_size, 'Unknown') AS company_size,
    COUNT(*)                          AS total_candidates,
    SUM(target)                       AS seeking_change,
    ROUND(AVG(target) * 100, 2)      AS turnover_rate_pct
FROM hr
GROUP BY company_size
ORDER BY turnover_rate_pct DESC
"""
con.execute(company_size_turnover).fetchdf()

,company_size,total_candidates,seeking_change,turnover_rate_pct
0,Unknown,5938,2410.0,40.59
1,10/49,1471,344.0,23.39
2,10000+,2019,385.0,19.07
3,5000-9999,563,102.0,18.12
4,50-99,3083,545.0,17.68
5,500-999,877,152.0,17.33
6,<10,1308,224.0,17.13
7,100-500,2571,415.0,16.14
8,1000-4999,1328,200.0,15.06


Among companies with known size, the 10/49 band shows the highest rate (23.39%). However, the Unknown group (likely freelancers and contractors) shows the highest overall rate at 40.59%.

### Turnover rate by years of experience
Let's look at the correlation of experience and turnover.

In [15]:
experience_turnover = """
SELECT
    COALESCE(experience, 'Unknown') AS experience,
    COUNT(*) AS total_candidates,
    SUM(target) AS seeking_change,
    ROUND(AVG(target) * 100, 2) AS turnover_rate_pct
FROM hr
GROUP BY experience
ORDER BY
    CASE
        WHEN experience = '<1' THEN 0
        WHEN experience = '>20' THEN 21
        WHEN experience IS NULL THEN 99
        ELSE TRY_CAST(experience AS INTEGER)
    END
"""
experience_turnover_quiery = con.execute(experience_turnover).fetchdf()
display(experience_turnover_quiery)

# Cross-check against pandas
exp_map = {'<1': 0, '>20': 21, **{str(i): i for i in range(1, 21)}}
check_exp = (
    train.assign(experience=train['experience'].fillna('Unknown'))
         .groupby('experience')['target']
         .agg(total='count', seekers='sum', rate='mean')
         .assign(rate=lambda d: (d['rate'] * 100).round(2))
         .assign(_sort=lambda d: d.index.map(
             lambda x: exp_map.get(x, 99)))
         .sort_values('_sort')
         .drop(columns='_sort')
)
print('Pandas cross-check:')
display(check_exp)

,experience,total_candidates,seeking_change,turnover_rate_pct
0,<1,522,237.0,45.40
1,1,549,233.0,42.44
2,2,1127,374.0,33.19
3,3,1354,478.0,35.30
4,4,1403,457.0,32.57
5,5,1430,412.0,28.81
6,6,1216,343.0,28.21
7,7,1028,303.0,29.47
8,8,802,195.0,24.31
9,9,980,213.0,21.73


Pandas cross-check:


,total,seekers,rate
experience,,,
<1,522,237.0,45.40
1,549,233.0,42.44
2,1127,374.0,33.19
3,1354,478.0,35.30
4,1403,457.0,32.57
5,1430,412.0,28.81
6,1216,343.0,28.21
7,1028,303.0,29.47
8,802,195.0,24.31


Attrition intent peaks at 1-4 years and declines from year 5 onward.

### Turnover rate by time since last job change

Let’s take a look at how the timing since the last job change relates to job turnover.

In [10]:
last_job_change_turnover = """
SELECT
    COALESCE(last_new_job, 'Unknown') AS last_new_job,
    COUNT(*)                          AS total_candidates,
    SUM(target)                       AS seeking_change,
    ROUND(AVG(target) * 100, 2)      AS turnover_rate_pct
FROM hr
GROUP BY last_new_job
ORDER BY
    CASE
        WHEN last_new_job = 'never' THEN 99
        WHEN last_new_job = '>4'    THEN 5
        WHEN last_new_job IS NULL   THEN 100
        ELSE TRY_CAST(last_new_job AS INTEGER)
    END
"""
con.execute(last_job_change_turnover).fetchdf()

,last_new_job,total_candidates,seeking_change,turnover_rate_pct
0,1,8040,2125.0,26.43
1,2,2900,700.0,24.14
2,3,1024,231.0,22.56
3,4,1029,228.0,22.16
4,>4,3290,600.0,18.24
5,never,2452,739.0,30.14
6,Unknown,423,154.0,36.41


Seeking rate generally declines with time since the last change, with one important exception: candidates who have never changed jobs show an elevated rate (30.14%), likely because they are in their first role and now testing the market.

### Rank cities by active job-seekers
Let's examine which cities have the biggest pools of actively job-seeking candidates.

In [11]:
city_active_seekers = """
SELECT
    city,
    city_development_index,
    COUNT(*) AS total_candidates,
    SUM(target) AS active_seekers,
    ROUND(AVG(target) * 100, 2) AS turnover_rate_pct,
    RANK() OVER (ORDER BY SUM(target) DESC) AS rank_by_seekers
FROM hr
GROUP BY city, city_development_index
QUALIFY RANK() OVER (ORDER BY SUM(target) DESC) <= 15
ORDER BY rank_by_seekers
"""
con.execute(city_active_seekers).fetchdf()

,city,city_development_index,total_candidates,active_seekers,turnover_rate_pct,rank_by_seekers
0,city_21,0.624,2702,1597.0,59.10,1
1,city_103,0.920,4355,928.0,21.31,2
2,city_160,0.920,845,199.0,23.55,3
3,city_16,0.910,1533,179.0,11.68,4
4,city_11,0.550,247,147.0,59.51,5
5,city_114,0.926,1336,133.0,9.96,6
6,city_73,0.754,280,74.0,26.43,7
7,city_100,0.887,275,65.0,23.64,8
8,city_90,0.698,197,62.0,31.47,9
9,city_136,0.897,586,61.0,10.41,10


The cities with the highest job-seeking rate tend to have the lowest CDI values - the four cities above 50% all have CDI below 0.63.

### Running total by training hours

In [12]:
training_hours = """
WITH training_hours_buckets AS (
    SELECT
        CASE
            WHEN training_hours < 50 THEN '01 — 0-49 hrs'
            WHEN training_hours < 100 THEN '02 — 50-99 hrs'
            WHEN training_hours < 150 THEN '03 — 100-149 hrs'
            WHEN training_hours < 200 THEN '04 — 150-199 hrs'
            ELSE '05 — 200+ hrs'
        END AS training_bucket,
        target
    FROM hr
)
SELECT
    training_bucket,
    COUNT(*) AS candidates,
    SUM(target) AS job_seekers,
    ROUND(AVG(target)*100, 2) AS turnover_rate_pct,
    SUM(COUNT(*)) OVER (
        ORDER BY training_bucket
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM training_hours_buckets
GROUP BY training_bucket
ORDER BY training_bucket
"""
con.execute(training_hours).fetchdf()

,training_bucket,candidates,job_seekers,turnover_rate_pct,running_total
0,01 — 0-49 hrs,9914,2520.0,25.42,9914.0
1,02 — 50-99 hrs,5263,1309.0,24.87,15177.0
2,03 — 100-149 hrs,2159,541.0,25.06,17336.0
3,04 — 150-199 hrs,997,241.0,24.17,18333.0
4,05 — 200+ hrs,825,166.0,20.12,19158.0


Training hours do not separate seekers from non-seekers at any bucket.

### Workforce segmentation — risk profile comparison
Let's check what does the high-risk employee profile look like vs. the stable employee.  

In [13]:
risk_profile = """
WITH risk_segments AS (
    SELECT *,
        CASE WHEN target = 1 THEN 'High Risk' ELSE 'Low Risk' END AS risk_label
    FROM hr
),
segment_summary AS (
    SELECT
        risk_label,
        COUNT(*)                              AS total,
        ROUND(AVG(training_hours), 1)         AS avg_training_hrs,
        ROUND(AVG(city_development_index), 4) AS avg_cdi,
        ROUND(MIN(city_development_index), 4) AS min_cdi,
        ROUND(MAX(city_development_index), 4) AS max_cdi
    FROM risk_segments
    GROUP BY risk_label
)
SELECT * FROM segment_summary ORDER BY risk_label DESC
"""
con.execute(risk_profile).fetchdf()

,risk_label,total,avg_training_hrs,avg_cdi,min_cdi,max_cdi
0,Low Risk,14381,66.1,0.8531,0.448,0.949
1,High Risk,4777,63.1,0.7557,0.448,0.949


High-risk employees are from cities with lower average CDI (~0.75) compared to low-risk employees (~0.83). Training hours are similar between groups, so it doesn't correlate with turnover.

## 4. Key Findings

* PhD holders have the lowest attrition rate among statistically meaningful groups ; Graduate the highest
* Missing company type group has the highest seeking rate - likely freelancers
* Employees from 10/49 person companies show the highest attrition rate
* Attrition peaks at 1–4 years of experience, then steadily declines
* Top seeker cities have lower CDI
* Training hours do not separate seekers from non-seekers at any bucket
* High-risk profile: lower CDI (0.75 vs 0.85), similar training hours